In [ ]:
import numpy as np
import subprocess
from pathlib import Path

In [ ]:
%run mesh_plotter.ipynb

In [ ]:
def slice_mesh_by_growth(mesh, max_growth_index, has_colour=True):
    """
    Return a partial mesh containing only faces deposited up to max_growth_index

    :param mesh: A shell, septa or siphuncle mesh
    :param max_growth_index: Extract up to this number of growth steps
    :param has_colour: True if the mesh contains colour values
    """

    if has_colour:
        X, Y, Z, C, I, J, K, face_growth_index = mesh

        face_growth_index = np.array(face_growth_index)
        mask = face_growth_index <= max_growth_index

        return (
            X, Y, Z, C,
            list(np.array(I)[mask]),
            list(np.array(J)[mask]),
            list(np.array(K)[mask]),
            face_growth_index[mask]
        )

    else:
        X, Y, Z, I, J, K, face_growth_index = mesh

        face_growth_index = np.array(face_growth_index)
        mask = face_growth_index <= max_growth_index

        return (
            X, Y, Z,
            list(np.array(I)[mask]),
            list(np.array(J)[mask]),
            list(np.array(K)[mask]),
            face_growth_index[mask]
        )

In [ ]:
def get_growth_frame_indices(shell_mesh, n_frames=120):
    """
    Choose evenly spaced shell growth indices for animation frames

    :param shell_mesh: Shell mesh
    :param n_frames: Number of frames in the movie
    """

    face_growth_index = shell_mesh[-1]

    min_growth = int(np.min(face_growth_index))
    max_growth = int(np.max(face_growth_index))

    return np.linspace(
        min_growth,
        max_growth,
        n_frames
    ).astype(int)

In [ ]:
def build_growth_frame_meshes(
    growth_index,
    shell_mesh,
    chamber_mesh=None,
    siphuncle_mesh=None
):
    """
    Slice shell, chamber, and siphuncle meshes to a given growth index

    :param growth_index: Number of growth steps to extract up to
    :param shell_mesh: Tuple returned by build_shell_mesh
    :param chamber_mesh: Optional chamber mesh returned by build_chamber_septa
    :param siphuncle_mesh: Optional siphuncle mesh returned by build_siphuncle_mesh
    """

    partial_shell = slice_mesh_by_growth(
        shell_mesh,
        max_growth_index=growth_index,
        has_colour=True
    )

    partial_chambers = None
    if chamber_mesh is not None:
        partial_chambers = slice_mesh_by_growth(
            chamber_mesh,
            max_growth_index=growth_index,
            has_colour=True
        )

    partial_siphuncle = None
    if siphuncle_mesh is not None:
        partial_siphuncle = slice_mesh_by_growth(
            siphuncle_mesh,
            max_growth_index=growth_index,
            has_colour=False
        )

    return partial_shell, partial_chambers, partial_siphuncle

In [ ]:
def render_growth_animation_frames(
    plotting_options,
    shell_mesh,
    output_dir,
    chamber_mesh=None,
    siphuncle_mesh=None,
    view="presentation",
    n_frames=120
):
    """
    Render a sequence of PNG frames showing shell growth

    :param plotting_options: Plotting options for the transparent shell view
    :param shell_mesh: Tuple returned by build_shell_mesh
    :param chamber_mesh: Optional chamber mesh returned by build_chamber_septa
    :param siphuncle_mesh: Optional siphuncle mesh returned by build_siphuncle_mesh
    :param view: Viewpoint for the frames
    :param n_frames: Number of frames to save
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    frame_growth_indices = get_growth_frame_indices(
        shell_mesh,
        n_frames=n_frames
    )

    frame_paths = []

    for frame_number, growth_index in enumerate(frame_growth_indices):
        partial_shell, partial_chambers, partial_siphuncle = build_growth_frame_meshes(
            growth_index,
            shell_mesh,
            chamber_mesh=chamber_mesh,
            siphuncle_mesh=siphuncle_mesh
        )

        png_path = output_dir / f"{frame_number:04d}.png"

        plot_and_save_shell_mesh(
            plotting_options,
            partial_shell,
            str(png_path),
            chamber_mesh=partial_chambers,
            siphuncle_mesh=partial_siphuncle,
            viewpoint=view
        )

        frame_paths.append(png_path)

    return frame_paths

In [ ]:
def render_movie_from_frames(
    frames_folder,
    output_path,
    fps = 24,
    start_number = 0,
    crf = 18,
    preset = "slow"
):
    """
    Stitch a set of frame images together as a movie using ffmpeg. The individual frames must be saved
    to the frames folder and named according to this pattern:

    %04d.png

    e.g. 0000.png, 0001.png, ...
    
    :param frames_folder: Path to the folder containing the frame images
    :param output_path: Output movie file path
    :param fps: Frames per second in the movie
    :param crf: Encoding constant rate factor, smaller -> higher quality -> larger file
    """
    frames_folder = Path(frames_folder)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    input_pattern = frames_folder / "%04d.png"

    cmd = [
        "ffmpeg",
        "-loglevel", "error",
        "-y",
        "-framerate", str(fps),
        "-start_number", str(start_number),
        "-i", str(input_pattern),
        "-c:v", "libx264",
        "-crf", str(crf),
        "-preset", preset,
        "-pix_fmt", "yuv420p",
        str(output_path),
    ]

    subprocess.run(cmd, check=True)